# Attack-Aware False Data Injection Detection in Railway SCADA

## Cleaned, reproducible evaluation notebook

This notebook evaluates statistical, temporal, and replay false-data-injection attacks using detector architectures matched to their expected signatures.

### Core methodological rules

- All scalers and thresholds are fitted on clean training data only.
- VAE reconstruction uses deterministic latent means (`z_mean`) during evaluation.
- MAD is treated as a thresholding procedure, not as a separate detector.
- One canonical results table is generated at the end of the notebook.
- CUSUM is reported as a classical baseline.
- Replay robustness is limited to the perturbations explicitly tested.


## Notebook map

| Section | Research purpose | Main outputs |
|---|---|---|
| 0–2 | Reproducible data preparation | Data audit and leakage-safe scaling |
| 3 | Clean-data baseline | Reconstruction distributions, PCA and t-SNE |
| 4–5 | Statistical attacks | Intensity analysis and feature susceptibility |
| 6–8 | Temporal attacks | LSTM-VAE and Transformer-VAE evaluation |
| 9–10 | Replay attacks | Stagnation detector, perturbed replay and CUSUM baseline |
| 11 | Canonical evaluation | Final metrics, confidence intervals and attack-family comparison |
| 12 | Export | Models, scalers and canonical tables |


## Revision log

This cleaned version integrates:

- deterministic VAE inference using `z_mean`;
- one MAD-thresholded reconstruction pipeline per VAE model;
- standardized detector names;
- removal of duplicate VAE/MAD result rows;
- removal of redundant visualizations;
- consistent training-only threshold calibration;
- a canonical final results table with fixed operating thresholds;
- bootstrap confidence intervals for the principal 5% experiments;
- accurate labeling of the final comparison as attack-family-specific; and
- removal of legacy-result export from the publication workflow.


## Section 0: Environment setup and reproducibility

In [ ]:
# Install / upgrade key dependencies
!pip install -q tensorflow scikit-learn pandas numpy matplotlib seaborn scipy openpyxl statsmodels


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings, os, json
warnings.filterwarnings('ignore')

# ── Deep learning ──────────────────────────────────────────
import tensorflow as tf
import keras
import keras.ops as kops  # Keras 3 backend-agnostic ops
from tensorflow.keras.layers import (Input, Dense, Lambda, LSTM, RepeatVector,
                                      TimeDistributed, Dropout, BatchNormalization)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import backend as K

# ── ML / stats ─────────────────────────────────────────────
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.model_selection import KFold
from sklearn.metrics import (confusion_matrix, classification_report,
                              precision_score, recall_score, f1_score,
                              roc_auc_score, roc_curve)
import statsmodels.stats.contingency_tables as smct
from scipy import stats

# ── Reproducibility ────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow {tf.__version__} | GPU: {tf.config.list_physical_devices('GPU')}")


## Section 1: Data loading

The notebook uses:

- `WRM_F.csv` for static railway geometry measurements.
- `RawD.xlsx` for longitudinal/temporal measurements.

Update the Google Drive paths in the next cell where necessary.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Update these paths if your files are in a sub-folder ───
BASE_PATH = '/content/drive/My Drive/'   # <-- change if needed
STATIC_FILE  = BASE_PATH + 'WRM_F.csv'
TEMPORAL_FILE = BASE_PATH + 'RawD.xlsx'


In [ ]:
# ── Load raw data ──────────────────────────────────────────
raw_df = pd.read_csv(STATIC_FILE)

# Rename BFI label columns only — do NOT rename BFI Value_1/2
# (renaming both creates duplicate column names, breaking pd.to_numeric)
raw_df = raw_df.rename(columns={
    'BFI L':   'BFI_L_Cat',      # category label — not used
    'BFI C ':  'BFI_C_Cat',      # category label — not used
    'BFI R ':  'BFI_R_Value',    # numeric BFI right value
    'BFI Value_1': 'BFI_L_Value', # numeric BFI left value
    'BFI Value_2': 'BFI_C_Value'  # numeric BFI centre value
})
print(f'Raw shape: {raw_df.shape}')

# ── Feature selection (domain knowledge + variance filter) ─
STATIC_FEATURES = ['LProf62','RProf62','LAlign62','RAlign62','Gage',
                   'Warp62','BFI_L_Value','BFI_C_Value','BFI_R_Value',
                   'Crosslevel','TotalLatForce','Left Cant','Right Cant','Speed']

# Safety check: no duplicate column names
dupes = [c for c in STATIC_FEATURES if raw_df.columns.tolist().count(c) > 1]
if dupes:
    raise ValueError(f'Duplicate columns detected: {dupes}. Check rename dict.')

# Coerce numeric, drop nulls
for col in STATIC_FEATURES:
    raw_df[col] = pd.to_numeric(raw_df[col], errors='coerce')
df_static = raw_df[STATIC_FEATURES].dropna().reset_index(drop=True)
print(f'After feature selection & dropna: {df_static.shape}')
print(df_static.describe().round(3))


## Section 2: Data preparation and quality audit

This stage performs feature selection, numeric conversion, missing-value handling, correlation screening and scaling. The retained variables are later attacked one at a time to quantify feature-specific susceptibility.

In [ ]:
# ── Remove highly correlated features (|r| > 0.85) ─────────
corr = df_static.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
to_drop = [c for c in upper.columns if any(upper[c] > 0.85)]
df_static = df_static.drop(columns=to_drop)
print(f"Dropped (high correlation): {to_drop}")
print(f"Final features ({df_static.shape[1]}): {list(df_static.columns)}")

STATIC_COLS = list(df_static.columns)


In [ ]:
# ── Scale ──────────────────────────────────────────────────
scaler_static = StandardScaler()
X_static = scaler_static.fit_transform(df_static)
X_static_df = pd.DataFrame(X_static, columns=STATIC_COLS)
print("Scaled shape:", X_static_df.shape)

# ── Quick EDA ──────────────────────────────────────────────
fig, axes = plt.subplots(2, len(STATIC_COLS)//2 + len(STATIC_COLS)%2,
                          figsize=(18, 5))
for ax, col in zip(axes.ravel(), STATIC_COLS):
    ax.hist(df_static[col], bins=40, edgecolor='none', alpha=0.7)
    ax.set_title(col, fontsize=8); ax.set_yticks([])
for ax in axes.ravel()[len(STATIC_COLS):]:
    ax.set_visible(False)
plt.suptitle("Feature Distributions — Clean Static Data", y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
# Reproducible data-quality summary for reporting
data_audit = pd.DataFrame({
    'item': [
        'Raw samples', 'Raw columns', 'Prepared samples',
        'Retained static features', 'Missing values after preparation',
        'Duplicate rows after preparation', 'Scaling method'
    ],
    'value': [
        len(raw_df), raw_df.shape[1], len(df_static),
        len(STATIC_COLS), int(df_static.isna().sum().sum()),
        int(df_static.duplicated().sum()), 'StandardScaler'
    ]
})
display(data_audit)

print('Retained static features:')
print(STATIC_COLS)

# Correlation matrix is descriptive only; it is not used as an exposure score.
plt.figure(figsize=(10, 8))
sns.heatmap(df_static.corr(), center=0, cmap='coolwarm', square=True)
plt.title('Correlation Matrix of Retained Static Features')
plt.tight_layout()
plt.show()

## Section 3: Clean-data baseline

The clean baseline establishes normal reconstruction behaviour before any FDI is introduced. The VAE produces reconstruction errors, while MAD provides a robust, training-derived anomaly score and threshold. Isolation Forest is retained as a classical static baseline.

### 3.1 Static detector definitions

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  VAE Architecture — Keras 3 / TF 2.16+
#  Fix: keras.random.normal (kops.random_normal does not exist in Keras 3)
# ─────────────────────────────────────────────────────────────────

class Sampling(keras.layers.Layer):
    """Reparametrisation trick: z = mu + exp(0.5 * log_var) * epsilon."""
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch   = kops.shape(z_mean)[0]
        dim     = kops.shape(z_mean)[1]
        epsilon = keras.random.normal(shape=(batch, dim))   # Keras 3 correct API
        return z_mean + kops.exp(0.5 * z_log_var) * epsilon


class VAEModel(keras.Model):
    """VAE wrapper — ELBO loss via self.add_loss() so .fit() works cleanly."""
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self._enc = encoder
        self._dec = decoder

    def call(self, x, training=False):
        z_mean, z_log_var, z = self._enc(x, training=training)
        x_hat  = self._dec(z, training=training)
        recon  = kops.mean(kops.sum(kops.square(x - x_hat), axis=1))
        kl     = -0.5 * kops.mean(
                     kops.sum(1 + z_log_var - kops.square(z_mean)
                              - kops.exp(z_log_var), axis=1))
        self.add_loss(recon + kl)
        return x_hat

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            'encoder': keras.saving.serialize_keras_object(self._enc),
            'decoder': keras.saving.serialize_keras_object(self._dec),
        })
        return cfg

    @classmethod
    def from_config(cls, config):
        enc = keras.saving.deserialize_keras_object(config.pop('encoder'))
        dec = keras.saving.deserialize_keras_object(config.pop('decoder'))
        return cls(encoder=enc, decoder=dec, **config)


def build_vae(input_dim, latent_dim=7):
    # ── Encoder ──────────────────────────────────────────────
    enc_input  = keras.Input(shape=(input_dim,), name='enc_input')
    h          = keras.layers.Dense(128, activation='relu')(enc_input)
    h          = keras.layers.BatchNormalization()(h)
    h          = keras.layers.Dropout(0.2)(h)
    h          = keras.layers.Dense(64, activation='relu')(h)
    h          = keras.layers.BatchNormalization()(h)
    z_mean     = keras.layers.Dense(latent_dim, name='z_mean')(h)
    z_log_var  = keras.layers.Dense(latent_dim, name='z_log_var')(h)
    z          = Sampling()([z_mean, z_log_var])
    encoder    = keras.Model(enc_input, [z_mean, z_log_var, z], name='encoder')

    # ── Decoder ──────────────────────────────────────────────
    dec_input  = keras.Input(shape=(latent_dim,), name='dec_input')
    d          = keras.layers.Dense(64, activation='relu')(dec_input)
    d          = keras.layers.BatchNormalization()(d)
    d          = keras.layers.Dropout(0.2)(d)
    d          = keras.layers.Dense(128, activation='relu')(d)
    dec_output = keras.layers.Dense(input_dim, activation='linear')(d)
    decoder    = keras.Model(dec_input, dec_output, name='decoder')

    vae = VAEModel(encoder, decoder, name='vae')
    vae.compile(optimizer=keras.optimizers.Adam(1e-3))
    return vae, encoder, decoder


# ─────────────────────────────────────────────────────────────────
#  Reconstruction + scoring helpers (leak-free)
#
#  - mad_score_from_train : MAD calibrated on CLEAN TRAINING errors only
#  - train_percentile_threshold : threshold from training distribution only
#  - cv_threshold REMOVED (used test labels — leakage)
# ─────────────────────────────────────────────────────────────────

def vae_reconstruction_error(encoder, decoder, X, feature_weights=None):
    """Deterministic reconstruction error using the latent posterior mean."""
    z_mean, _, _ = encoder.predict(X, verbose=0)
    X_hat = decoder.predict(z_mean, verbose=0)
    squared_error = (X - X_hat) ** 2
    if feature_weights is not None:
        squared_error = feature_weights * squared_error
    return np.mean(squared_error, axis=1)

def mad_score_from_train(train_errors, test_errors):
    """MAD calibrated on CLEAN TRAINING errors only — no leakage."""
    median = np.median(train_errors)
    mad    = np.median(np.abs(train_errors - median))
    return np.abs(test_errors - median) / (mad + 1e-8)

def train_percentile_threshold(train_scores, percentile=95):
    """Threshold derived from training distribution only — no label leakage."""
    return np.percentile(train_scores, percentile)

def evaluate(y_true, y_pred, label='Model'):
    p   = precision_score(y_true, y_pred, zero_division=0)
    r   = recall_score(y_true, y_pred, zero_division=0)
    f1  = f1_score(y_true, y_pred, zero_division=0)
    acc = np.mean(y_true == y_pred)
    print(f'{label:50s}  P={p:.3f}  R={r:.3f}  F1={f1:.3f}  Acc={acc:.3f}')
    return {'label': label, 'precision': p, 'recall': r, 'f1': f1, 'accuracy': acc}

def plot_confusion(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(4,3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Normal','Anomaly'],
                yticklabels=['Normal','Anomaly'])
    plt.title(title); plt.ylabel('True'); plt.xlabel('Predicted')
    plt.tight_layout(); plt.show()


In [ ]:
# Helper functions are defined in Cell 11:
#   vae_reconstruction_error, mad_score_from_train,
#   train_percentile_threshold, evaluate, plot_confusion
# This cell is intentionally left as a placeholder to preserve cell numbering.
pass


### 3.2 Train the static VAE and classical baseline

In [ ]:
input_dim_static = X_static.shape[1]

# ── Feature weights: emphasise safety-critical sensors ──────
fw_static = np.array([1.5 if c in ('Gage','LAlign62','RAlign62','Warp62','Crosslevel')
                       else 1.0 for c in STATIC_COLS])

# ── Train VAE on clean data ──────────────────────────────────
vae_s, enc_s, dec_s = build_vae(input_dim_static, latent_dim=7)
cb = [EarlyStopping(patience=10, restore_best_weights=True),
      ReduceLROnPlateau(patience=5, factor=0.5, verbose=0)]
history = vae_s.fit(X_static, X_static,
                    epochs=100, batch_size=64,
                    validation_split=0.15,
                    callbacks=cb, verbose=0)
print("VAE training complete. Final val loss:", round(history.history['val_loss'][-1], 4))

# ── Training curve ───────────────────────────────────────────
plt.figure(figsize=(7,3))
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend()
plt.title('Static VAE — Training Curve'); plt.tight_layout(); plt.show()

In [ ]:
# ── Isolation Forest on clean data ──────────────────────────
iso_s = IsolationForest(contamination=0.05, random_state=SEED)
iso_s.fit(X_static)

# ── Pre-injection reconstruction errors ─────────────────────
err_vae_pre    = vae_reconstruction_error(enc_s, dec_s, X_static, fw_static)
iso_scores_pre = -iso_s.score_samples(X_static)

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for ax, scores, name in zip(axes,
    [iso_scores_pre, err_vae_pre],
    ['Isolation Forest', 'VAE Reconstruction Error']):
    ax.hist(scores, bins=60, alpha=0.7, edgecolor='none')
    ax.set_title(f'{name} — Pre-FDI')
    ax.set_xlabel('Score')
plt.tight_layout(); plt.show()


In [ ]:
# VAE with MAD thresholding: one detector pipeline
train_vae_errors = vae_reconstruction_error(enc_s, dec_s, X_static, fw_static)
train_mad_scores = mad_score_from_train(train_vae_errors, train_vae_errors)
mad_threshold = train_percentile_threshold(train_mad_scores, percentile=95)

print(f"VAE with MAD thresholding — training threshold: {mad_threshold:.4f}")

plt.figure(figsize=(7, 3.5))
plt.hist(train_mad_scores, bins=60, alpha=0.75)
plt.axvline(mad_threshold, linestyle='--', linewidth=2,
            label='95th-percentile MAD-score threshold')
plt.xlabel('Robust MAD score')
plt.ylabel('Frequency')
plt.title('Clean-Data Robust Reconstruction-Score Distribution')
plt.legend()
plt.tight_layout()
plt.show()


### 3.3 Low-dimensional visualization of clean behaviour

In [ ]:
# ── PCA & t-SNE on clean latent space ───────────────────────
z_pre = enc_s.predict(X_static, verbose=0)[0]   # z_mean

pca   = PCA(n_components=2)
z_pca = pca.fit_transform(z_pre)

tsne   = TSNE(n_components=2, random_state=SEED, perplexity=30)
z_tsne = tsne.fit_transform(z_pre[:3000])   # subsample for speed

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(z_pca[:,0], z_pca[:,1], s=2, alpha=0.3, c='steelblue')
axes[0].set_title('PCA — Latent Space (Pre-FDI)')
axes[1].scatter(z_tsne[:,0], z_tsne[:,1], s=2, alpha=0.3, c='darkorange')
axes[1].set_title('t-SNE — Latent Space (Pre-FDI)')
plt.tight_layout(); plt.show()


## Section 4: Statistical FDI generation

Three statistical manipulation mechanisms are simulated: additive spikes, multiplicative scaling and Gaussian noise. Attack prevalence is evaluated at 2%, 5% and 10%.

In [ ]:
def inject_static(df, injection_pct=0.05, seed=SEED):
    """
    Inject synthetic FDI into Gage, Warp62, Crosslevel.
    Uses numpy array operations — avoids pandas .loc indexing pitfalls.
    """
    rng        = np.random.default_rng(seed)
    n          = len(df)
    n_anomalies = int(n * injection_pct)
    window_len  = 46
    n_windows   = max(1, n_anomalies // window_len)
    spread      = (n - n_windows * window_len) // (n_windows + 1)

    arr   = df.values.copy().astype(float)   # numpy array — no .loc needed
    y     = np.zeros(n, dtype=int)
    cols  = list(df.columns)
    attack_col_idx = [cols.index(c) for c in ['Gage','Warp62','Crosslevel'] if c in cols]

    for i in range(n_windows):
        start = (i + 1) * spread + i * window_len
        end   = min(start + window_len, n)
        y[start:end] = 1

        method = rng.choice(['spike', 'multiply', 'noise'])
        for ci in attack_col_idx:
            std = arr[:, ci].std()
            if method == 'spike':
                arr[start:end, ci] += rng.choice([-1, 1]) * rng.uniform(2, 3) * std
            elif method == 'multiply':
                arr[start:end, ci] *= rng.uniform(1.5, 2.0)
            else:
                arr[start:end, ci] += rng.normal(0, 3 * std, size=(end - start))

    df_inj = pd.DataFrame(arr, columns=cols)
    print(f'[inject_static] pct={injection_pct*100:.0f}%  '
          f'windows={n_windows}  anomalies={int(y.sum())}/{n}')
    return df_inj, y

# ── Inject at 5% (main experiment) ──────────────────────────
df_inj5, y_true5 = inject_static(df_static, injection_pct=0.05)
X_inj5    = scaler_static.transform(df_inj5)
X_inj5_df = pd.DataFrame(X_inj5, columns=STATIC_COLS)


In [ ]:
# Illustrative validation plot: clean versus statistically attacked signal
_plot_feature = 'Gage' if 'Gage' in df_static.columns else df_static.columns[0]
_plot_idx = df_static.columns.get_loc(_plot_feature)
_plot_n = min(500, len(df_static))
plt.figure(figsize=(11, 3.5))
plt.plot(df_static[_plot_feature].iloc[:_plot_n].to_numpy(), label='Clean', linewidth=1.2)
plt.plot(df_inj5[_plot_feature].iloc[:_plot_n].to_numpy(), label='Attacked', linewidth=1.0, alpha=0.8)
plt.xlabel('Sample index')
plt.ylabel(_plot_feature)
plt.title(f'Statistical FDI Validation — {_plot_feature}')
plt.legend()
plt.tight_layout()
plt.show()

## Section 5: Statistical attack detection and feature susceptibility

In [ ]:
def run_static_detection(X_train, X_inj, y_true, enc, dec, iso, fw, label_prefix='5pct'):
    """Evaluate statistical detectors with thresholds fitted on clean data."""
    train_iso = -iso.score_samples(X_train)
    test_iso = -iso.score_samples(X_inj)
    iso_threshold = train_percentile_threshold(train_iso, 95)

    train_err = vae_reconstruction_error(enc, dec, X_train, fw)
    test_err = vae_reconstruction_error(enc, dec, X_inj, fw)
    train_mad = mad_score_from_train(train_err, train_err)
    test_mad = mad_score_from_train(train_err, test_err)
    vae_threshold = train_percentile_threshold(train_mad, 95)

    return {
        'Isolation Forest': evaluate(y_true, (test_iso > iso_threshold).astype(int),
                                     f'{label_prefix} Isolation Forest'),
        'VAE with MAD thresholding': evaluate(y_true, (test_mad > vae_threshold).astype(int),
                                              f'{label_prefix} VAE with MAD thresholding'),
    }

static_results = {}
static_results['5pct'] = run_static_detection(
    X_static, X_inj5, y_true5, enc_s, dec_s, iso_s, fw_static, '5pct')


In [ ]:
# Confusion matrices for the primary 5% statistical experiment
train_iso = -iso_s.score_samples(X_static)
test_iso = -iso_s.score_samples(X_inj5)
iso_threshold = np.percentile(train_iso, 95)

train_err = vae_reconstruction_error(enc_s, dec_s, X_static, fw_static)
test_err = vae_reconstruction_error(enc_s, dec_s, X_inj5, fw_static)
train_mad = mad_score_from_train(train_err, train_err)
test_mad = mad_score_from_train(train_err, test_err)
vae_threshold = np.percentile(train_mad, 95)

plot_confusion(y_true5, (test_iso > iso_threshold).astype(int),
               'Isolation Forest — Statistical FDI (5%)')
plot_confusion(y_true5, (test_mad > vae_threshold).astype(int),
               'VAE with MAD Thresholding — Statistical FDI (5%)')


In [ ]:
# ── PCA / t-SNE post-injection ───────────────────────────────
z_post = enc_s.predict(X_inj5, verbose=0)[0]
z_post_pca  = pca.transform(z_post[:3000])
z_post_tsne = TSNE(n_components=2, random_state=SEED, perplexity=30).fit_transform(z_post[:3000])
y_sub = y_true5[:3000]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, Z, title in zip(axes,
    [z_post_pca, z_post_tsne],
    ['PCA — Post-FDI (5%)', 't-SNE — Post-FDI (5%)']):
    ax.scatter(Z[y_sub==0, 0], Z[y_sub==0, 1], s=2, alpha=0.3, c='steelblue', label='Normal')
    ax.scatter(Z[y_sub==1, 0], Z[y_sub==1, 1], s=8, alpha=0.8, c='crimson', label='Injected')
    ax.set_title(title); ax.legend(markerscale=4, fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Statistical attack-intensity analysis: 2%, 5% and 10%
injection_levels = [0.02, 0.05, 0.10]
all_static = {}

for pct in injection_levels:
    df_inj, y_inj = inject_static(df_static, injection_pct=pct)
    X_inj = scaler_static.transform(df_inj)
    all_static[pct] = run_static_detection(
        X_static, X_inj, y_inj, enc_s, dec_s, iso_s, fw_static,
        f'{int(pct*100)}pct')

models_ = ['Isolation Forest', 'VAE with MAD thresholding']
metrics_ = ['f1', 'precision', 'recall']
x_labels = ['2%', '5%', '10%']
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, metric in zip(axes, metrics_):
    for model_name in models_:
        values = [all_static[pct][model_name][metric] for pct in injection_levels]
        ax.plot(x_labels, values, marker='o', label=model_name)
    ax.set_title(metric.upper()); ax.set_ylim(0,1)
    ax.set_xlabel('Injected sample proportion'); ax.legend(fontsize=8)
plt.suptitle('Statistical FDI Detection Across Attack Prevalence')
plt.tight_layout(); plt.show()


### 5.1 Controlled one-feature statistical attacks

**Operational definition.** Feature susceptibility is the difficulty of detecting an attack when only one railway measurement is manipulated. A feature is more difficult to protect when its attacks produce lower recall, F1-score, or ranking performance under the same injection protocol.


In [ ]:
def inject_one_feature_static_scaled(
    X_clean,
    feature_idx,
    injection_pct=0.05,
    magnitude_z=2.5,
    window_len=46,
    seed=SEED
):
    """
    Controlled one-feature attack in STANDARDIZED space.

    All features receive the same perturbation magnitude in z-score units.
    The same seed produces the same attacked windows for every feature,
    enabling a fair feature-to-feature comparison.
    """
    rng = np.random.default_rng(seed)
    X_inj = np.array(X_clean, copy=True)
    n = len(X_inj)
    target_n = max(1, int(round(n * injection_pct)))
    n_windows = max(1, int(np.ceil(target_n / window_len)))

    # Select approximately evenly spaced, non-overlapping windows.
    available = max(1, n - n_windows * window_len)
    gap = available // (n_windows + 1)

    y = np.zeros(n, dtype=int)
    attacked = 0
    for w in range(n_windows):
        start = (w + 1) * gap + w * window_len
        end = min(start + window_len, n)

        # Trim the final window so prevalence is matched as closely as possible.
        remaining = target_n - attacked
        if remaining <= 0:
            break
        end = min(end, start + remaining)

        direction = rng.choice([-1.0, 1.0])
        X_inj[start:end, feature_idx] += direction * magnitude_z
        y[start:end] = 1
        attacked += end - start

    return X_inj, y


def static_feature_sensitivity(
    X_clean,
    feature_names,
    enc,
    dec,
    iso,
    feature_weights,
    injection_pct=0.05,
    magnitude_z=2.5,
    seed=SEED
):
    """
    Evaluate IF, VAE, and VAE+MAD under matched one-feature attacks.
    Thresholds are calibrated from clean-score distributions only.
    """
    train_vae = vae_reconstruction_error(enc, dec, X_clean, feature_weights)
    train_iso = -iso.score_samples(X_clean)
    train_mad = mad_score_from_train(train_vae, train_vae)

    thresholds = {
        'IF': train_percentile_threshold(train_iso, 95),
        'VAE': train_percentile_threshold(train_vae, 95),
        'VAE+MAD': train_percentile_threshold(train_mad, 95),
    }

    rows = []
    for j, feature in enumerate(feature_names):
        X_inj, y_true = inject_one_feature_static_scaled(
            X_clean,
            feature_idx=j,
            injection_pct=injection_pct,
            magnitude_z=magnitude_z,
            seed=seed
        )

        score_vae = vae_reconstruction_error(enc, dec, X_inj, feature_weights)
        score_iso = -iso.score_samples(X_inj)
        score_mad = mad_score_from_train(train_vae, score_vae)

        score_map = {'IF': score_iso, 'VAE': score_vae, 'VAE+MAD': score_mad}
        for model_name, scores in score_map.items():
            y_pred = (scores > thresholds[model_name]).astype(int)
            rows.append({
                'Feature': feature,
                'Model': model_name,
                'Attack prevalence': injection_pct,
                'Magnitude (z)': magnitude_z,
                'Precision': precision_score(y_true, y_pred, zero_division=0),
                'Recall': recall_score(y_true, y_pred, zero_division=0),
                'F1': f1_score(y_true, y_pred, zero_division=0),
                'False positive rate': (
                    ((y_pred == 1) & (y_true == 0)).sum()
                    / max(1, (y_true == 0).sum())
                )
            })

    return pd.DataFrame(rows)


# Preliminary controlled diagnostic.
# For the final paper, rerun this cell on the HELD-OUT static test split only.
static_feature_df = static_feature_sensitivity(
    X_static,
    STATIC_COLS,
    enc_s,
    dec_s,
    iso_s,
    fw_static,
    injection_pct=0.05,
    magnitude_z=2.5,
    seed=SEED
)

display(static_feature_df.sort_values(['Model', 'Recall'], ascending=[True, False]))

# Feature-wise recall heatmap
static_recall_pivot = static_feature_df.pivot(
    index='Feature', columns='Model', values='Recall'
)
plt.figure(figsize=(8, max(5, 0.35 * len(STATIC_COLS))))
sns.heatmap(static_recall_pivot, annot=True, fmt='.2f', vmin=0, vmax=1, cmap='viridis')
plt.title('Static Feature-Wise FDI Detectability\nMatched prevalence and standardized magnitude')
plt.xlabel('Detector')
plt.ylabel('Attacked feature')
plt.tight_layout()
plt.show()

# Rank potentially stealthier features using mean recall across detectors.
static_feature_rank = (
    static_feature_df.groupby('Feature', as_index=False)
    .agg(
        Mean_Recall=('Recall', 'mean'),
        Mean_F1=('F1', 'mean'),
        Mean_FPR=('False positive rate', 'mean')
    )
    .sort_values('Mean_Recall')
)
print('Lower recall indicates a potentially stealthier attacked feature:')
display(static_feature_rank)


## Section 6: Temporal data preparation

Temporal sequences are constructed only from positions observed across the full longitudinal period. The train/test split is performed before temporal scaling to reduce leakage across positions.

In [ ]:
# ── Features shared with WRM_F ──────────────────────────────
TEMPORAL_FEATURES = ['GAGE', 'XLEVEL', 'WARP_62',
                     'SUF_L_62', 'SUF_R_62',
                     'ALI_L_62', 'ALI_R_62']
TEMPORAL_COLS_RENAME = {
    'GAGE': 'Gage', 'XLEVEL': 'Crosslevel', 'WARP_62': 'Warp62',
    'SUF_L_62': 'LProf62', 'SUF_R_62': 'RProf62',
    'ALI_L_62': 'LAlign62', 'ALI_R_62': 'RAlign62'
}

xl     = pd.ExcelFile(TEMPORAL_FILE)
sheets = [s for s in xl.sheet_names if s != 'Chart']
print(f'Inspection months: {len(sheets)}  ({sheets[0]} → {sheets[-1]})')

monthly_dfs = {}
for sheet in sheets:
    df = xl.parse(sheet)
    df = df[['MILE', 'FEET'] + TEMPORAL_FEATURES].dropna()
    df = df.rename(columns=TEMPORAL_COLS_RENAME)
    # ── KEY FIX: cast to int to avoid '64_1.0' vs '64_1' mismatch ──
    df['pos_key'] = (df['MILE'].astype(int).astype(str) + '_' +
                     df['FEET'].round(0).astype(int).astype(str))
    # Drop duplicate positions within same month (keep first)
    df = df.drop_duplicates('pos_key')
    monthly_dfs[sheet] = df.set_index('pos_key')

print('Sample shape per sheet:', next(iter(monthly_dfs.values())).shape)
# Sanity-check: show a few keys from first sheet
print('Sample keys:', sorted(monthly_dfs[sheets[0]].index)[:5])


In [ ]:
# ── Find positions present in ALL 28 months ─────────────────
common_keys = set(monthly_dfs[sheets[0]].index)
for s in sheets[1:]:
    common_keys &= set(monthly_dfs[s].index)
common_keys = sorted(common_keys)
print(f"Track positions in all months: {len(common_keys)}")

# ── Build tensor: (n_positions, n_months, n_features) ────────
FEAT_COLS = list(TEMPORAL_COLS_RENAME.values())
n_pos    = len(common_keys)
n_months = len(sheets)
n_feat   = len(FEAT_COLS)

X_temporal_raw = np.zeros((n_pos, n_months, n_feat))
for t, sheet in enumerate(sheets):
    df_t = monthly_dfs[sheet].loc[common_keys, FEAT_COLS]
    X_temporal_raw[:, t, :] = df_t.values

print(f"Temporal tensor shape: {X_temporal_raw.shape}")
print(f"  = ({n_pos} positions × {n_months} months × {n_feat} features)")


In [ ]:
# ── Preserve raw tensor — scaling happens AFTER train/test split ──────
# (Fitting scaler on all positions before split would leak test statistics
#  into the training normalisation.)
X_temporal = X_temporal_raw.copy()
print("Raw temporal tensor (unscaled):", X_temporal.shape)
print("Scaler will be fitted on training positions only in the next section.")

# ── Visualise raw signal for a few positions ──────────────────
fig, axes = plt.subplots(2, len(FEAT_COLS), figsize=(18, 5))
sample_positions = np.random.choice(n_pos, 10, replace=False)
months_label = [s.replace('_', '/') for s in sheets]
for j, feat in enumerate(FEAT_COLS):
    ax0, ax1 = axes[0, j], axes[1, j]
    for pos in sample_positions:
        ax0.plot(X_temporal[pos, :, j], alpha=0.4, lw=0.8)
    ax0.set_title(feat, fontsize=8); ax0.set_xticks([])
    ax1.plot(X_temporal[:, :, j].mean(axis=0), color='steelblue', lw=2)
    ax1.fill_between(range(n_months),
        X_temporal[:, :, j].mean(axis=0) - X_temporal[:, :, j].std(axis=0),
        X_temporal[:, :, j].mean(axis=0) + X_temporal[:, :, j].std(axis=0),
        alpha=0.2)
    ax1.set_xticks(range(0, n_months, 4))
    ax1.set_xticklabels(months_label[::4], rotation=45, fontsize=6)
axes[0, 0].set_ylabel('Sample pos.', fontsize=8)
axes[1, 0].set_ylabel('Mean ± std', fontsize=8)
plt.suptitle('Temporal Signals — 28-Month Track Geometry (raw, unscaled)', fontsize=12)
plt.tight_layout(); plt.show()


## Section 7: Temporal detector architectures

### 7.1 LSTM-VAE

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  LSTMVAEModel subclass — same Keras 3 pattern as VAEModel
# ─────────────────────────────────────────────────────────────────
class LSTMVAEModel(keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self._enc = encoder
        self._dec = decoder


    def get_config(self):
        config = super().get_config()
        config.update({
            'encoder': keras.saving.serialize_keras_object(self._enc),
            'decoder': keras.saving.serialize_keras_object(self._dec),
        })
        return config

    @classmethod
    def from_config(cls, config):
        enc = keras.saving.deserialize_keras_object(config.pop('encoder'))
        dec = keras.saving.deserialize_keras_object(config.pop('decoder'))
        return cls(encoder=enc, decoder=dec, **config)
    def call(self, x, training=False):
        zm, zlv, z = self._enc(x, training=training)
        x_hat = self._dec(z, training=training)
        recon = kops.mean(
                    kops.mean(kops.sum(kops.square(x - x_hat), axis=-1), axis=-1))
        kl    = -0.5 * kops.mean(
                    kops.sum(1 + zlv - kops.square(zm) - kops.exp(zlv), axis=-1))
        self.add_loss(recon + kl)
        return x_hat

def build_lstm_vae(seq_len, n_features, latent_dim=16, lstm_units=64):
    # ── Encoder ──────────────────────────────────────────────
    enc_input = keras.Input(shape=(seq_len, n_features), name='lstm_enc_in')
    h = keras.layers.Bidirectional(
            keras.layers.LSTM(lstm_units, return_sequences=True))(enc_input)
    h = keras.layers.Dropout(0.1)(h)
    h = keras.layers.LSTM(lstm_units // 2)(h)
    h = keras.layers.BatchNormalization()(h)
    z_mean    = keras.layers.Dense(latent_dim, name='z_mean')(h)
    z_log_var = keras.layers.Dense(latent_dim, name='z_log_var')(h)
    z         = Sampling()([z_mean, z_log_var])
    encoder   = keras.Model(enc_input, [z_mean, z_log_var, z], name='lstm_encoder')

    # ── Decoder ──────────────────────────────────────────────
    dec_input  = keras.Input(shape=(latent_dim,), name='lstm_dec_in')
    d          = keras.layers.Dense(lstm_units // 2, activation='relu')(dec_input)
    d          = keras.layers.RepeatVector(seq_len)(d)
    d          = keras.layers.LSTM(lstm_units // 2, return_sequences=True)(d)
    d          = keras.layers.Dropout(0.1)(d)
    d          = keras.layers.LSTM(lstm_units, return_sequences=True)(d)
    dec_output = keras.layers.TimeDistributed(
                     keras.layers.Dense(n_features), name='recon')(d)
    decoder    = keras.Model(dec_input, dec_output, name='lstm_decoder')

    lstm_vae = LSTMVAEModel(encoder, decoder, name='lstm_vae')
    lstm_vae.compile(optimizer=keras.optimizers.Adam(1e-3))
    return lstm_vae, encoder, decoder

lstm_vae, lstm_enc, lstm_dec = build_lstm_vae(
    seq_len=n_months, n_features=n_feat, latent_dim=16, lstm_units=64)
lstm_vae.summary()


In [ ]:
# ── Split BEFORE scaling — prevents test-position leakage ───────────
from sklearn.model_selection import train_test_split

idx_all = np.arange(n_pos)
idx_train, idx_test = train_test_split(idx_all, test_size=0.2, random_state=SEED)

X_train_raw = X_temporal_raw[idx_train]
X_test_raw  = X_temporal_raw[idx_test]

# Fit scaler on training positions only, then transform both splits
scaler_t = StandardScaler()
X_train_t = scaler_t.fit_transform(
    X_train_raw.reshape(-1, n_feat)).reshape(X_train_raw.shape)
X_test_t  = scaler_t.transform(
    X_test_raw.reshape(-1, n_feat)).reshape(X_test_raw.shape)

print(f"Train: {X_train_t.shape}   Test: {X_test_t.shape}")
print("Scaler fitted on training positions only — no leakage ✓")

cb_lstm = [
    EarlyStopping(patience=15, restore_best_weights=True),
    ReduceLROnPlateau(patience=7, factor=0.5, verbose=0)
]
history_lstm = lstm_vae.fit(
    X_train_t, X_train_t,
    epochs=150, batch_size=128,
    validation_split=0.15,
    callbacks=cb_lstm, verbose=1
)

plt.figure(figsize=(7,3))
plt.plot(history_lstm.history['loss'], label='train')
plt.plot(history_lstm.history['val_loss'], label='val')
plt.xlabel('Epoch'); plt.ylabel('ELBO Loss'); plt.legend()
plt.title('LSTM-VAE — Training Curve'); plt.tight_layout(); plt.show()


### 7.2 Transformer-VAE

In [ ]:
# ─────────────────────────────────────────────────────────────────
#  Positional Encoding — sinusoidal, fixed (no training needed)
# ─────────────────────────────────────────────────────────────────
class PositionalEncoding(keras.layers.Layer):
    """Adds sinusoidal positional encoding to (batch, T, d_model) tensors."""
    def __init__(self, seq_len, d_model, **kwargs):
        super().__init__(**kwargs)
        self.seq_len = seq_len
        self.d_model = d_model
        pos   = np.arange(seq_len, dtype=np.float32)[:, np.newaxis]
        i     = np.arange(d_model, dtype=np.float32)[np.newaxis, :]
        angle = pos / np.power(10000.0, (2 * (i // 2)) / d_model)
        angle[:, 0::2] = np.sin(angle[:, 0::2])
        angle[:, 1::2] = np.cos(angle[:, 1::2])
        self._pe = tf.constant(angle[np.newaxis], dtype=tf.float32)  # (1,T,d)

    def call(self, x):
        return x + self._pe[:, :kops.shape(x)[1], :]

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'seq_len': self.seq_len, 'd_model': self.d_model})
        return cfg


# ─────────────────────────────────────────────────────────────────
#  Transformer-VAE wrapper  (same subclass pattern as LSTMVAEModel)
# ─────────────────────────────────────────────────────────────────
class TransformerVAEModel(keras.Model):
    """
    Transformer encoder + LSTM decoder VAE.
    Multi-head self-attention captures global temporal context so that
    any month inconsistent with the rest of the sequence raises loss —
    critical for one-shot and short persistent attacks.
    """
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self._enc = encoder
        self._dec = decoder

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            'encoder': keras.saving.serialize_keras_object(self._enc),
            'decoder': keras.saving.serialize_keras_object(self._dec),
        })
        return cfg

    @classmethod
    def from_config(cls, config):
        enc = keras.saving.deserialize_keras_object(config.pop('encoder'))
        dec = keras.saving.deserialize_keras_object(config.pop('decoder'))
        return cls(encoder=enc, decoder=dec, **config)

    def call(self, x, training=False):
        zm, zlv, z = self._enc(x, training=training)
        x_hat = self._dec(z, training=training)
        recon = kops.mean(
                    kops.mean(kops.sum(kops.square(x - x_hat), axis=-1), axis=-1))
        kl    = -0.5 * kops.mean(
                    kops.sum(1 + zlv - kops.square(zm) - kops.exp(zlv), axis=-1))
        self.add_loss(recon + kl)
        return x_hat


def build_transformer_vae(seq_len, n_features, latent_dim=16,
                           d_model=64, n_heads=4, n_layers=2, dropout=0.1):
    """
    Encoder : Dense projection → Positional Encoding
              → N × (MultiHeadAttention + LayerNorm + FFN + LayerNorm)
              → GlobalAveragePooling → BatchNorm → z_mean / z_log_var
    Decoder : Dense → RepeatVector → LSTM × 2 → TimeDistributed Dense
    """
    # ── Encoder ──────────────────────────────────────────────
    enc_input = keras.Input(shape=(seq_len, n_features), name='trans_enc_in')
    x  = keras.layers.Dense(d_model)(enc_input)          # project features → d_model
    x  = PositionalEncoding(seq_len, d_model)(x)         # add positional context

    for _ in range(n_layers):
        attn = keras.layers.MultiHeadAttention(
                   num_heads=n_heads, key_dim=d_model // n_heads,
                   dropout=dropout)(x, x)                # self-attention
        x    = keras.layers.LayerNormalization(epsilon=1e-6)(x + attn)
        ff   = keras.layers.Dense(d_model * 2, activation='relu')(x)
        ff   = keras.layers.Dropout(dropout)(ff)
        ff   = keras.layers.Dense(d_model)(ff)
        x    = keras.layers.LayerNormalization(epsilon=1e-6)(x + ff)

    h         = keras.layers.GlobalAveragePooling1D()(x)  # pool over time
    h         = keras.layers.BatchNormalization()(h)
    z_mean    = keras.layers.Dense(latent_dim, name='z_mean')(h)
    z_log_var = keras.layers.Dense(latent_dim, name='z_log_var')(h)
    z         = Sampling()([z_mean, z_log_var])
    encoder   = keras.Model(enc_input, [z_mean, z_log_var, z], name='trans_encoder')

    # ── Decoder (same as LSTM-VAE) ────────────────────────────
    dec_input  = keras.Input(shape=(latent_dim,), name='trans_dec_in')
    d          = keras.layers.Dense(d_model // 2, activation='relu')(dec_input)
    d          = keras.layers.RepeatVector(seq_len)(d)
    d          = keras.layers.LSTM(d_model // 2, return_sequences=True)(d)
    d          = keras.layers.Dropout(dropout)(d)
    d          = keras.layers.LSTM(d_model, return_sequences=True)(d)
    dec_output = keras.layers.TimeDistributed(
                     keras.layers.Dense(n_features))(d)
    decoder    = keras.Model(dec_input, dec_output, name='trans_decoder')

    trans_vae = TransformerVAEModel(encoder, decoder, name='trans_vae')
    trans_vae.compile(optimizer=keras.optimizers.Adam(1e-3))
    return trans_vae, encoder, decoder


trans_vae, trans_enc, trans_dec = build_transformer_vae(
    seq_len=n_months, n_features=n_feat,
    latent_dim=16, d_model=64, n_heads=4, n_layers=2)
trans_vae.summary()


In [ ]:
# ── Train Transformer-VAE on clean sequences ────────────────
print('Training Transformer-VAE...')
cb_trans = [
    EarlyStopping(patience=15, restore_best_weights=True),
    ReduceLROnPlateau(patience=7, factor=0.5, verbose=0)
]
history_trans = trans_vae.fit(
    X_train_t, X_train_t,
    epochs=150, batch_size=128,
    validation_split=0.15,
    callbacks=cb_trans, verbose=1
)

plt.figure(figsize=(7,3))
plt.plot(history_trans.history['loss'],     label='train')
plt.plot(history_trans.history['val_loss'], label='val')
plt.xlabel('Epoch'); plt.ylabel('ELBO Loss'); plt.legend()
plt.title('Transformer-VAE — Training Curve')
plt.tight_layout(); plt.show()


## Section 8: Temporal FDI generation and detection

Persistent attacks alter several consecutive time steps, while one-shot attacks alter a single time step. Each sequence receives a binary attack label for sequence-level evaluation.

In [ ]:
def inject_temporal(X_seq, injection_pct=0.05, attack_type='persistent',
                    persist_months=6, seed=SEED):
    """
    Inject FDI into temporal sequences.
    attack_type='persistent' : tamper `persist_months` consecutive months
    attack_type='one_shot'   : tamper a single random month
    Returns (X_injected, y_seq_labels, y_step_labels)
    """
    rng      = np.random.default_rng(seed)
    n_seq, n_t, n_f = X_seq.shape
    n_attack = max(1, int(n_seq * injection_pct))

    # Guard: persist_months must be < n_t
    persist_months = min(persist_months, n_t - 1)

    X_inj      = X_seq.copy()
    y_seq      = np.zeros(n_seq, dtype=int)
    y_step     = np.zeros((n_seq, n_t), dtype=int)
    attack_idx = rng.choice(n_seq, size=n_attack, replace=False)
    y_seq[attack_idx] = 1

    feat_idx = [0, 1, 2]   # Gage, Crosslevel, Warp62

    for idx in attack_idx:
        if attack_type == 'persistent':
            max_start = n_t - persist_months
            t_start   = int(rng.integers(0, max(1, max_start)))
            t_end     = t_start + persist_months
            y_step[idx, t_start:t_end] = 1
            method = rng.choice(['spike', 'drift', 'noise'])
            for f in feat_idx:
                if method == 'spike':
                    amp = float(rng.choice([-1, 1])) * rng.uniform(2, 3)
                    X_inj[idx, t_start:t_end, f] += amp
                elif method == 'drift':
                    drift = np.linspace(0, rng.uniform(1.5, 3.0), persist_months)
                    X_inj[idx, t_start:t_end, f] += drift
                else:
                    X_inj[idx, t_start:t_end, f] += rng.normal(0, 2.5, persist_months)
        else:   # one_shot
            t_atk = int(rng.integers(0, n_t))
            y_step[idx, t_atk] = 1
            method = rng.choice(['spike', 'noise'])
            for f in feat_idx:
                if method == 'spike':
                    X_inj[idx, t_atk, f] += float(rng.choice([-1, 1])) * rng.uniform(3, 5)
                else:
                    X_inj[idx, t_atk, f] += rng.normal(0, 4)

    print(f'[inject_temporal] type={attack_type}  pct={injection_pct*100:.0f}%  '
          f'attacked={int(y_seq.sum())}/{n_seq}')
    return X_inj, y_seq, y_step

# ── Preview all injection configs ───────────────────────────
for atk_type in ['persistent', 'one_shot']:
    for pct in [0.02, 0.05, 0.10]:
        Xi, yi, _ = inject_temporal(X_test_t, injection_pct=pct, attack_type=atk_type)
        print(f'  {atk_type} {int(pct*100)}%: shape={Xi.shape}  attacks={yi.sum()}')


In [ ]:
# Illustrative temporal attack validation (created after the injection cell)
# The code is defensive because variable names can differ when individual cells are rerun.
for _name in ['X_inj_p5', 'X_inj_o5']:
    if _name in globals():
        _X = globals()[_name]
        _seq = 0
        _feat = 0
        plt.figure(figsize=(9, 3))
        plt.plot(X_test_t[_seq, :, _feat], marker='o', label='Clean')
        plt.plot(_X[_seq, :, _feat], marker='o', label=_name)
        plt.xlabel('Time step')
        plt.ylabel(TEMP_FEATURES[_feat] if 'TEMP_FEATURES' in globals() else f'Feature {_feat}')
        plt.title(f'Temporal Attack Validation — {_name}')
        plt.legend()
        plt.tight_layout()
        plt.show()
        break

In [ ]:
def temporal_reconstruction_error(encoder, decoder, X_seq, reduction='max'):
    """Deterministic sequence reconstruction error using z_mean."""
    z_mean, _, _ = encoder.predict(X_seq, verbose=0)
    X_hat = decoder.predict(z_mean, verbose=0)
    per_step = np.mean((X_seq - X_hat) ** 2, axis=-1)
    if reduction == 'max':
        return np.max(per_step, axis=-1)
    if reduction == 'mean':
        return np.mean(per_step, axis=-1)
    raise ValueError("reduction must be 'max' or 'mean'")

lstm_reconstruction_error = temporal_reconstruction_error

def run_temporal_detection(X_train, X_test_inj, y_seq_true, enc, dec, label):
    """Evaluate one VAE detector with MAD-based thresholding."""
    train_err = temporal_reconstruction_error(enc, dec, X_train)
    test_err = temporal_reconstruction_error(enc, dec, X_test_inj)
    train_mad = mad_score_from_train(train_err, train_err)
    test_mad = mad_score_from_train(train_err, test_err)
    threshold = train_percentile_threshold(train_mad, 95)
    prediction = (test_mad > threshold).astype(int)
    result = evaluate(y_seq_true, prediction, label)
    result['threshold'] = threshold
    return {label: result}, test_err, test_mad

X_inj_p5, y_p5, _ = inject_temporal(X_test_t, 0.05, 'persistent')
res_p5, err_p5, mad_p5 = run_temporal_detection(
    X_train_t, X_inj_p5, y_p5, lstm_enc, lstm_dec,
    'LSTM-VAE with MAD thresholding — Persistent 5%')

X_inj_o5, y_o5, _ = inject_temporal(X_test_t, 0.05, 'one_shot')
res_o5, err_o5, mad_o5 = run_temporal_detection(
    X_train_t, X_inj_o5, y_o5, lstm_enc, lstm_dec,
    'LSTM-VAE with MAD thresholding — One-shot 5%')


In [ ]:
# ── Reconstruction error distributions ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, err, y_lbl, title in zip(axes,
    [err_p5, err_o5],
    [y_p5,   y_o5],
    ['Persistent Attack (5%)', 'One-Shot Attack (5%)']):
    ax.hist(err[y_lbl==0], bins=50, alpha=0.6, label='Normal', color='steelblue')
    ax.hist(err[y_lbl==1], bins=50, alpha=0.6, label='Injected', color='crimson')
    ax.set_title(f'LSTM-VAE Recon Error — {title}')
    ax.set_xlabel('Reconstruction Error'); ax.legend()
plt.tight_layout(); plt.show()

# ── Latent space post-injection ──────────────────────────────
z_inj_p = lstm_enc.predict(X_inj_p5, verbose=0)[0]
pca_t   = PCA(n_components=2).fit_transform(z_inj_p)
plt.figure(figsize=(6,4))
plt.scatter(pca_t[y_p5==0,0], pca_t[y_p5==0,1], s=4, alpha=0.3, label='Normal')
plt.scatter(pca_t[y_p5==1,0], pca_t[y_p5==1,1], s=15, alpha=0.8, c='crimson', label='Injected')
plt.title('LSTM-VAE Latent Space — Persistent FDI (5%)')
plt.legend(markerscale=3); plt.tight_layout(); plt.show()


In [ ]:
temporal_results = {}
for attack_type in ['persistent', 'one_shot']:
    temporal_results[attack_type] = {}
    for pct in [0.02, 0.05, 0.10]:
        X_inj, y_inj, _ = inject_temporal(X_test_t, pct, attack_type)
        label = ('LSTM-VAE with MAD thresholding — '
                 f"{attack_type.replace('_',' ').title()} {int(pct*100)}%")
        result, _, _ = run_temporal_detection(
            X_train_t, X_inj, y_inj, lstm_enc, lstm_dec, label)
        temporal_results[attack_type][pct] = result


In [ ]:
# Transformer-VAE detection — primary 5% attacks
X_inj_tp5, y_tp5, _ = inject_temporal(X_test_t, 0.05, 'persistent')
res_tp5, err_tp5, mad_tp5 = run_temporal_detection(
    X_train_t, X_inj_tp5, y_tp5, trans_enc, trans_dec,
    'Transformer-VAE with MAD thresholding — Persistent 5%')

X_inj_to5, y_to5, _ = inject_temporal(X_test_t, 0.05, 'one_shot')
res_to5, err_to5, mad_to5 = run_temporal_detection(
    X_train_t, X_inj_to5, y_to5, trans_enc, trans_dec,
    'Transformer-VAE with MAD thresholding — One-shot 5%')

fig, axes = plt.subplots(1, 2, figsize=(13,4))
for ax, err_l, err_t, y_lbl, title in zip(
    axes, [err_p5,err_o5], [err_tp5,err_to5], [y_p5,y_o5],
    ['Persistent 5%','One-shot 5%']):
    ax.hist(err_l[y_lbl==0], bins=40, alpha=0.4, label='Normal: LSTM-VAE')
    ax.hist(err_l[y_lbl==1], bins=40, alpha=0.4, label='Attacked: LSTM-VAE')
    ax.hist(err_t[y_lbl==0], bins=40, alpha=0.4, label='Normal: Transformer-VAE')
    ax.hist(err_t[y_lbl==1], bins=40, alpha=0.4, label='Attacked: Transformer-VAE')
    ax.set_title(f'Reconstruction Error — {title}')
    ax.set_xlabel('Reconstruction error'); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

trans_results = {}
for attack_type in ['persistent','one_shot']:
    trans_results[attack_type] = {}
    for pct in [0.02,0.05,0.10]:
        X_inj, y_inj, _ = inject_temporal(X_test_t, pct, attack_type)
        label = ('Transformer-VAE with MAD thresholding — '
                 f"{attack_type.replace('_',' ').title()} {int(pct*100)}%")
        result, _, _ = run_temporal_detection(
            X_train_t, X_inj, y_inj, trans_enc, trans_dec, label)
        trans_results[attack_type][pct] = result


### 8.1 Temporal feature-specific susceptibility

In [ ]:
def inject_one_feature_temporal(
    X_seq,
    feature_idx,
    injection_pct=0.05,
    attack_type='persistent',
    persist_months=6,
    magnitude_z=2.5,
    seed=SEED
):
    """
    Controlled temporal FDI against one feature at a time.

    Perturbation magnitude is expressed in standardized units.
    The same seed fixes attacked sequences and months across features.
    """
    rng = np.random.default_rng(seed)
    X_inj = np.array(X_seq, copy=True)
    n_seq, n_t, _ = X_inj.shape
    n_attack = max(1, int(round(n_seq * injection_pct)))

    attacked_idx = rng.choice(n_seq, size=n_attack, replace=False)
    y_seq = np.zeros(n_seq, dtype=int)
    y_step = np.zeros((n_seq, n_t), dtype=int)
    y_seq[attacked_idx] = 1

    persist_months = min(max(1, persist_months), n_t)

    for idx in attacked_idx:
        direction = rng.choice([-1.0, 1.0])

        if attack_type == 'persistent':
            max_start = n_t - persist_months
            start = int(rng.integers(0, max_start + 1))
            end = start + persist_months
            X_inj[idx, start:end, feature_idx] += direction * magnitude_z
            y_step[idx, start:end] = 1

        elif attack_type == 'one_shot':
            t = int(rng.integers(0, n_t))
            X_inj[idx, t, feature_idx] += direction * magnitude_z
            y_step[idx, t] = 1

        elif attack_type == 'gradual':
            max_start = n_t - persist_months
            start = int(rng.integers(0, max_start + 1))
            end = start + persist_months
            drift = np.linspace(
                magnitude_z / persist_months,
                magnitude_z,
                persist_months
            )
            X_inj[idx, start:end, feature_idx] += direction * drift
            y_step[idx, start:end] = 1

        else:
            raise ValueError(
                "attack_type must be 'persistent', 'one_shot', or 'gradual'"
            )

    return X_inj, y_seq, y_step


def temporal_feature_sensitivity(
    X_train,
    X_test,
    feature_names,
    model_specs,
    injection_pct=0.05,
    attack_types=('persistent', 'one_shot'),
    persist_months=6,
    magnitude_z=2.5,
    seed=SEED
):
    """
    model_specs: dictionary such as
      {'LSTM-VAE': (lstm_enc, lstm_dec),
       'Transformer-VAE': (trans_enc, trans_dec)}
    """
    rows = []

    # Precompute clean thresholds once per model.
    model_thresholds = {}
    model_train_err = {}
    for model_name, (encoder, decoder) in model_specs.items():
        train_err = lstm_reconstruction_error(encoder, decoder, X_train)
        train_mad = mad_score_from_train(train_err, train_err)
        model_train_err[model_name] = train_err
        model_thresholds[model_name] = {
            'Raw': train_percentile_threshold(train_err, 95),
            'MAD': train_percentile_threshold(train_mad, 95),
        }

    for attack_type in attack_types:
        for feature_idx, feature in enumerate(feature_names):
            X_inj, y_true, _ = inject_one_feature_temporal(
                X_test,
                feature_idx=feature_idx,
                injection_pct=injection_pct,
                attack_type=attack_type,
                persist_months=persist_months,
                magnitude_z=magnitude_z,
                seed=seed
            )

            for model_name, (encoder, decoder) in model_specs.items():
                test_err = lstm_reconstruction_error(encoder, decoder, X_inj)
                test_mad = mad_score_from_train(
                    model_train_err[model_name], test_err
                )

                for scoring, scores in [('Raw', test_err), ('MAD', test_mad)]:
                    threshold = model_thresholds[model_name][scoring]
                    y_pred = (scores > threshold).astype(int)
                    rows.append({
                        'Feature': feature,
                        'Attack type': attack_type,
                        'Model': model_name,
                        'Scoring': scoring,
                        'Attack prevalence': injection_pct,
                        'Magnitude (z)': magnitude_z,
                        'Precision': precision_score(
                            y_true, y_pred, zero_division=0
                        ),
                        'Recall': recall_score(
                            y_true, y_pred, zero_division=0
                        ),
                        'F1': f1_score(
                            y_true, y_pred, zero_division=0
                        ),
                        'False positive rate': (
                            ((y_pred == 1) & (y_true == 0)).sum()
                            / max(1, (y_true == 0).sum())
                        )
                    })

    return pd.DataFrame(rows)


temporal_feature_df = temporal_feature_sensitivity(
    X_train_t,
    X_test_t,
    FEAT_COLS,
    model_specs={
        'LSTM-VAE': (lstm_enc, lstm_dec),
        'Transformer-VAE': (trans_enc, trans_dec),
    },
    injection_pct=0.05,
    attack_types=('persistent', 'one_shot', 'gradual'),
    persist_months=6,
    magnitude_z=2.5,
    seed=SEED
)

display(
    temporal_feature_df.sort_values(
        ['Attack type', 'Model', 'Scoring', 'Recall'],
        ascending=[True, True, True, False]
    )
)

# Heatmaps use raw reconstruction-error scoring for a direct model comparison.
for attack_type in ['persistent', 'one_shot', 'gradual']:
    plot_df = temporal_feature_df[
        (temporal_feature_df['Attack type'] == attack_type)
        & (temporal_feature_df['Scoring'] == 'Raw')
    ]
    recall_pivot = plot_df.pivot(
        index='Feature', columns='Model', values='Recall'
    )

    plt.figure(figsize=(8, max(4, 0.55 * len(FEAT_COLS))))
    sns.heatmap(
        recall_pivot,
        annot=True,
        fmt='.2f',
        vmin=0,
        vmax=1,
        cmap='viridis'
    )
    plt.title(
        f'Temporal Feature-Wise Detectability — {attack_type.replace("_", " ").title()}'
    )
    plt.xlabel('Temporal detector')
    plt.ylabel('Attacked feature')
    plt.tight_layout()
    plt.show()

# Overall ranking: low recall means relatively stealthier under matched attacks.
temporal_feature_rank = (
    temporal_feature_df[
        temporal_feature_df['Scoring'] == 'Raw'
    ]
    .groupby(['Feature', 'Attack type'], as_index=False)
    .agg(
        Mean_Recall=('Recall', 'mean'),
        Mean_F1=('F1', 'mean')
    )
    .sort_values(['Attack type', 'Mean_Recall'])
)

print('Potentially stealthier temporal targets appear first within each attack type:')
display(temporal_feature_rank)


## Section 9: Replay attack generation and replay-aware detection

The replay experiments include exact replay, noisy replay, drift-perturbed replay, partial replay, and feature-restricted replay. These tests evaluate robustness only under the perturbations explicitly implemented below.


In [ ]:
# ── Layer 3: Stagnation Score for Replay (T0856) ───────────────────────

# Re-define inject_temporal_replay in case cell 36 was skipped
def inject_temporal_replay(X_seq, pct, replay_window=6, seed=SEED):
    rng      = np.random.default_rng(seed)
    n_seq, n_t, n_f = X_seq.shape
    n_attack = max(1, int(n_seq * pct))
    attack_indices = rng.choice(n_seq, n_attack, replace=False)
    X_inj = X_seq.copy()
    y     = np.zeros(n_seq, dtype=int)
    for idx in attack_indices:
        y[idx] = 1
        src_start = int(rng.integers(0, n_t // 2 - replay_window + 1))
        tgt_start = n_t - replay_window
        X_inj[idx, tgt_start:, :] = X_seq[idx, src_start:src_start + replay_window, :]
    return X_inj, y, attack_indices

def stagnation_score(X_seq, window=6):
    N, T, F = X_seq.shape
    scores  = np.zeros(N)
    for i, seq in enumerate(X_seq):
        last_win = seq[T - window:]
        min_mse  = np.inf
        for s in range(0, T - 2 * window + 1):
            mse = np.mean((last_win - seq[s : s + window]) ** 2)
            if mse < min_mse:
                min_mse = mse
        scores[i] = min_mse
    return scores

# Fresh replay injection at 5%
X_inj_r5, y_r5, _ = inject_temporal_replay(X_test_t, 0.05)

train_stag  = stagnation_score(X_train_t, window=6)
thresh_stag = np.percentile(train_stag, 5)

test_stag   = stagnation_score(X_inj_r5, window=6)
pred_stag   = (test_stag < thresh_stag).astype(int)

err_r5_lstm  = lstm_reconstruction_error(lstm_enc,  lstm_dec,  X_inj_r5)
err_r5_trans = lstm_reconstruction_error(trans_enc, trans_dec, X_inj_r5)

auc_stag = roc_auc_score(y_r5, -test_stag)
f1_stag  = f1_score(y_r5, pred_stag)
print(f"Stagnation | AUC={auc_stag:.3f}   F1={f1_stag:.3f}")

fig, ax = plt.subplots(figsize=(7, 5))
for scores, y_lbl, label, color, ls in [
    (err_r5_lstm,  y_r5, 'LSTM-VAE',             'crimson',   '--'),
    (err_r5_trans, y_r5, 'Transformer-VAE',             'darkgreen',  ':'),
    (-test_stag,   y_r5, 'Stagnation (proposed)', 'steelblue',  '-'),
]:
    fpr, tpr, _ = roc_curve(y_lbl, scores)
    auc = roc_auc_score(y_lbl, scores)
    ax.plot(fpr, tpr, lw=2, ls=ls, color=color, label=f'{label} (AUC={auc:.3f})')

ax.plot([0,1],[0,1],'k--', alpha=0.3, label='Random')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('Replay Attack Detection (5% FDI)\nLayer 3: Stagnation Score vs Reconstruction-Based')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── Perturbed Replay Variants ─────────────────────────────

import numpy as np
from sklearn.metrics import roc_auc_score

ATTACK_FEAT_IDX = [0, 1, 2]  # Gage, Crosslevel, Warp62

def inject_replay_noisy(X_seq, pct, noise_sigma=0.05, replay_window=6, seed=SEED):
    """Replay + Gaussian noise: xt = x_{t-k} + ε,  ε ~ N(0, noise_sigma²).
    noise_sigma is expressed as fraction of per-feature std from training."""
    rng = np.random.default_rng(seed)
    feat_std = X_seq.reshape(-1, X_seq.shape[-1]).std(axis=0)
    n_seq, n_t, n_f = X_seq.shape
    n_attack = max(1, int(n_seq * pct))
    attack_idx = rng.choice(n_seq, n_attack, replace=False)
    X_inj = X_seq.copy()
    y = np.zeros(n_seq, dtype=int)
    for idx in attack_idx:
        y[idx] = 1
        src_start = int(rng.integers(0, n_t // 2 - replay_window + 1))
        tgt_start = n_t - replay_window
        replayed = X_seq[idx, src_start:src_start + replay_window, :].copy()
        noise = rng.normal(0, noise_sigma * feat_std, replayed.shape)
        X_inj[idx, tgt_start:, :] = replayed + noise
    return X_inj, y

def inject_replay_drift(X_seq, pct, drift_rate=0.05, replay_window=6, seed=SEED):
    """Replay + linear drift: xt = x_{t-k} + δ·t.
    drift_rate is fraction of per-feature std per month."""
    rng = np.random.default_rng(seed)
    feat_std = X_seq.reshape(-1, X_seq.shape[-1]).std(axis=0)
    n_seq, n_t, n_f = X_seq.shape
    n_attack = max(1, int(n_seq * pct))
    attack_idx = rng.choice(n_seq, n_attack, replace=False)
    X_inj = X_seq.copy()
    y = np.zeros(n_seq, dtype=int)
    for idx in attack_idx:
        y[idx] = 1
        src_start = int(rng.integers(0, n_t // 2 - replay_window + 1))
        tgt_start = n_t - replay_window
        replayed = X_seq[idx, src_start:src_start + replay_window, :].copy()
        for t in range(replay_window):
            replayed[t] += drift_rate * feat_std * t
        X_inj[idx, tgt_start:, :] = replayed
    return X_inj, y

def inject_replay_partial(X_seq, pct, feat_idx=None, replay_window=6, seed=SEED):
    """Replay only selected features — attacker only spoofs Gage sensor."""
    if feat_idx is None:
        feat_idx = [0]   # Gage only
    rng = np.random.default_rng(seed)
    n_seq, n_t, n_f = X_seq.shape
    n_attack = max(1, int(n_seq * pct))
    attack_idx = rng.choice(n_seq, n_attack, replace=False)
    X_inj = X_seq.copy()
    y = np.zeros(n_seq, dtype=int)
    for idx in attack_idx:
        y[idx] = 1
        src_start = int(rng.integers(0, n_t // 2 - replay_window + 1))
        tgt_start = n_t - replay_window
        X_inj[idx, tgt_start:, feat_idx] = X_seq[idx, src_start:src_start + replay_window, feat_idx]
    return X_inj, y

# ── Run all variants at 5% FDI ──────────────────────────────────────────
variants = {
    'Exact replay':       inject_temporal_replay(X_test_t, 0.05),
    'Replay+noise σ=0.05': inject_replay_noisy(X_test_t, 0.05, noise_sigma=0.05),
    'Replay+noise σ=0.10': inject_replay_noisy(X_test_t, 0.05, noise_sigma=0.10),
    'Replay+noise σ=0.20': inject_replay_noisy(X_test_t, 0.05, noise_sigma=0.20),
    'Replay+drift δ=0.05': inject_replay_drift(X_test_t, 0.05, drift_rate=0.05),
    'Replay+drift δ=0.10': inject_replay_drift(X_test_t, 0.05, drift_rate=0.10),
    'Partial (Gage only)': inject_replay_partial(X_test_t, 0.05, feat_idx=[0]),
}

print(f"{'Variant':<28}  {'Stagnation':>11}  {'LSTM-VAE':>9}  {'Transformer-VAE':>10}")
print("-" * 65)
results_robust = {}
for name, (X_inj, y_inj, *_) in variants.items():
    stag  = stagnation_score(X_inj, window=6)
    e_l   = lstm_reconstruction_error(lstm_enc,  lstm_dec,  X_inj)
    e_t   = lstm_reconstruction_error(trans_enc, trans_dec, X_inj)
    auc_s = roc_auc_score(y_inj, -stag)
    auc_l = roc_auc_score(y_inj,  e_l)
    auc_t = roc_auc_score(y_inj,  e_t)
    results_robust[name] = (auc_s, auc_l, auc_t)
    print(f"{name:<28}  {auc_s:>11.3f}  {auc_l:>9.3f}  {auc_t:>10.3f}")

# ── Plot: AUC vs noise level ────────────────────────────────────────────
noise_levels = [0.0, 0.05, 0.10, 0.20]
stag_aucs = [results_robust['Exact replay'][0],
             results_robust['Replay+noise σ=0.05'][0],
             results_robust['Replay+noise σ=0.10'][0],
             results_robust['Replay+noise σ=0.20'][0]]
lstm_aucs = [results_robust['Exact replay'][1],
             results_robust['Replay+noise σ=0.05'][1],
             results_robust['Replay+noise σ=0.10'][1],
             results_robust['Replay+noise σ=0.20'][1]]
trans_aucs= [results_robust['Exact replay'][2],
             results_robust['Replay+noise σ=0.05'][2],
             results_robust['Replay+noise σ=0.10'][2],
             results_robust['Replay+noise σ=0.20'][2]]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(noise_levels, stag_aucs,  'b-o',  lw=2, label='Stagnation (proposed)')
ax.plot(noise_levels, lstm_aucs,  'r--s', lw=2, label='LSTM-VAE')
ax.plot(noise_levels, trans_aucs, 'g:^',  lw=2, label='Transformer-VAE')
ax.axhline(0.5, color='k', ls='--', alpha=0.3, label='Random')
ax.set_xlabel('Replay Noise σ (fraction of feature std)')
ax.set_ylabel('AUC')
ax.set_title('Stagnation Score Robustness to Adversarial Noise\n(Replay Attack, 5% FDI)')
ax.legend(); ax.grid(alpha=0.3)
ax.set_xlim(-0.01, 0.21); ax.set_ylim(0.4, 1.05)
plt.tight_layout(); plt.savefig('stagnation_robustness.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 10: Replay feature susceptibility

Replay is applied independently to individual railway measurements and to selected multi-feature combinations. This directly measures which signals are hardest to protect against replay manipulation.

In [ ]:
# ── Partial Replay Robustness Analysis ─────────────────────────────────
# Features: 0=Gage, 1=Crosslevel, 2=Warp62, 3=LProf62, 4=RProf62,
#           5=LAlign62, 6=RAlign62

FEAT_NAMES = ['Gage', 'Crosslevel', 'Warp62', 'LProf62', 'RProf62', 'LAlign62', 'RAlign62']

def inject_replay_partial(X_seq, pct, feat_idx, replay_window=6, seed=SEED):
    """Replay only specified feature indices."""
    rng = np.random.default_rng(seed)
    n_seq, n_t, n_f = X_seq.shape
    n_attack   = max(1, int(n_seq * pct))
    attack_idx = rng.choice(n_seq, n_attack, replace=False)
    X_inj = X_seq.copy()
    y     = np.zeros(n_seq, dtype=int)
    for idx in attack_idx:
        y[idx] = 1
        src_start = int(rng.integers(0, n_t // 2 - replay_window + 1))
        tgt_start = n_t - replay_window
        X_inj[idx, tgt_start:, feat_idx] = X_seq[idx, src_start:src_start + replay_window, feat_idx]
    return X_inj, y

def eval_stagnation(X_inj, y_inj, window=6):
    stag = stagnation_score(X_inj, window=window)
    return roc_auc_score(y_inj, -stag)

def eval_lstm(X_inj, y_inj):
    err = lstm_reconstruction_error(lstm_enc, lstm_dec, X_inj)
    return roc_auc_score(y_inj, err)

def eval_trans(X_inj, y_inj):
    err = lstm_reconstruction_error(trans_enc, trans_dec, X_inj)
    return roc_auc_score(y_inj, err)

# ── Single-feature replay ───────────────────────────────────────────────
print("SINGLE-FEATURE PARTIAL REPLAY (5% FDI)")
print(f"{'Feature':<12}  {'Stagnation':>11}  {'LSTM-VAE':>9}  {'Transformer-VAE':>10}")
print("-" * 48)

single_stag, single_lstm, single_trans = [], [], []
for i, name in enumerate(FEAT_NAMES):
    X_inj, y_inj = inject_replay_partial(X_test_t, 0.05, feat_idx=[i])
    a_s = eval_stagnation(X_inj, y_inj)
    a_l = eval_lstm(X_inj, y_inj)
    a_t = eval_trans(X_inj, y_inj)
    single_stag.append(a_s); single_lstm.append(a_l); single_trans.append(a_t)
    print(f"{name:<12}  {a_s:>11.3f}  {a_l:>9.3f}  {a_t:>10.3f}")

# ── Multi-feature combinations ──────────────────────────────────────────
from itertools import combinations

safety_feats = [0, 1, 2]   # Gage, Crosslevel, Warp62 — geometry-critical
combos = list(combinations(safety_feats, 2)) + \
         [(0,1,2)]          # all three safety features

print(f"\nMULTI-FEATURE PARTIAL REPLAY (safety-critical features, 5% FDI)")
print(f"{'Features':<22}  {'Stagnation':>11}  {'LSTM-VAE':>9}  {'Transformer-VAE':>10}")
print("-" * 58)

combo_labels, combo_stag, combo_lstm, combo_trans = [], [], [], []
for feat_idx in combos:
    label = '+'.join(FEAT_NAMES[i] for i in feat_idx)
    X_inj, y_inj = inject_replay_partial(X_test_t, 0.05, feat_idx=list(feat_idx))
    a_s = eval_stagnation(X_inj, y_inj)
    a_l = eval_lstm(X_inj, y_inj)
    a_t = eval_trans(X_inj, y_inj)
    combo_labels.append(label); combo_stag.append(a_s)
    combo_lstm.append(a_l);     combo_trans.append(a_t)
    print(f"{label:<22}  {a_s:>11.3f}  {a_l:>9.3f}  {a_t:>10.3f}")

# ── Heatmap: Stagnation AUC by feature scope ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: single-feature bar chart
ax = axes[0]
x  = np.arange(len(FEAT_NAMES))
bars = ax.bar(x, single_stag, color=['steelblue' if s >= 0.7 else 'coral' for s in single_stag],
              alpha=0.85, edgecolor='k', linewidth=0.5)
ax.plot(x, single_lstm,  'r--o', lw=1.5, ms=5, label='LSTM-VAE')
ax.plot(x, single_trans, 'g:^',  lw=1.5, ms=5, label='Transformer-VAE')
ax.axhline(0.5, color='k', ls=':', alpha=0.4)
ax.set_xticks(x); ax.set_xticklabels(FEAT_NAMES, rotation=30, ha='right', fontsize=9)
ax.set_ylim(0.4, 1.05); ax.set_ylabel('AUC'); ax.legend(fontsize=8)
ax.set_title('Single-Feature Partial Replay\nStagnation vs Baselines')
ax.grid(axis='y', alpha=0.3)

# Right: multi-feature bar chart
ax = axes[1]
x2 = np.arange(len(combo_labels))
ax.bar(x2, combo_stag, color=['steelblue' if s >= 0.7 else 'coral' for s in combo_stag],
       alpha=0.85, edgecolor='k', linewidth=0.5)
ax.plot(x2, combo_lstm,  'r--o', lw=1.5, ms=5, label='LSTM-VAE')
ax.plot(x2, combo_trans, 'g:^',  lw=1.5, ms=5, label='Transformer-VAE')
ax.axhline(0.5, color='k', ls=':', alpha=0.4)
ax.set_xticks(x2); ax.set_xticklabels(combo_labels, rotation=20, ha='right', fontsize=9)
ax.set_ylim(0.4, 1.05); ax.legend(fontsize=8)
ax.set_title('Multi-Feature Partial Replay\n(Safety-Critical Combinations)')
ax.grid(axis='y', alpha=0.3)

plt.suptitle('Stagnation Score Robustness — Partial Replay at 5% FDI', fontweight='bold')
plt.tight_layout()
plt.savefig('partial_replay_robustness.png', dpi=150, bbox_inches='tight')
plt.show()

### 10.1 CUSUM persistent-drift baseline

CUSUM is included as a classical sequential change-detection baseline. It is not presented as a proposed model.


In [ ]:
# ── Layer 2: CUSUM for Persistent Drift (T0856) ────────────────────────

def fit_cusum(X_train_seq):
    X_flat = X_train_seq.reshape(-1, X_train_seq.shape[-1])
    mu0    = X_flat.mean(axis=0)
    sigma  = X_flat.std(axis=0) + 1e-8
    return mu0, sigma

ATTACK_FEAT_IDX = [0, 1, 2]   # Gage, Crosslevel, Warp62 — features injected into

def cusum_score(X_seq, mu0, sigma, k_factor=0.5, feat_idx=ATTACK_FEAT_IDX):
    # Restrict to attacked features only — avoids noise dilution from clean features
    if feat_idx is not None:
        X_seq  = X_seq[:, :, feat_idx]
        mu0    = mu0[feat_idx]
        sigma  = sigma[feat_idx]
    scores = np.zeros(len(X_seq))
    for i, seq in enumerate(X_seq):
        S_pos = np.zeros(seq.shape[1])
        S_neg = np.zeros(seq.shape[1])
        peak  = 0.0
        for t in range(len(seq)):
            dev   = (seq[t] - mu0) / sigma
            S_pos = np.maximum(0.0, S_pos + dev - k_factor)
            S_neg = np.maximum(0.0, S_neg - dev - k_factor)
            peak  = max(peak, S_pos.max(), S_neg.max())
        scores[i] = peak
    return scores

# Fit on training data
mu0_cu, sigma_cu   = fit_cusum(X_train_t)
train_cu           = cusum_score(X_train_t, mu0_cu, sigma_cu)
thresh_cu          = np.percentile(train_cu, 95)

# Evaluate on persistent 5% test set (already created in cell 38)
test_cu_p5         = cusum_score(X_inj_p5, mu0_cu, sigma_cu)
pred_cu            = (test_cu_p5 > thresh_cu).astype(int)

auc_cu  = roc_auc_score(y_p5, test_cu_p5)
f1_cu   = f1_score(y_p5, pred_cu)
print(f"CUSUM  |  AUC={auc_cu:.3f}   F1={f1_cu:.3f}   threshold={thresh_cu:.3f}")

# ROC comparison: CUSUM vs LSTM-VAE vs Transformer-VAE on persistent drift
fig, ax = plt.subplots(figsize=(7, 5))
for scores, y_lbl, label, color, ls in [
    (err_p5,    y_p5,  'LSTM-VAE',  'crimson',   '--'),
    (err_tp5,   y_tp5, 'Transformer-VAE', 'darkgreen',  ':'),
    (test_cu_p5, y_p5, 'CUSUM baseline', 'steelblue', '-'),
]:
    fpr, tpr, _ = roc_curve(y_lbl, scores)
    auc = roc_auc_score(y_lbl, scores)
    ax.plot(fpr, tpr, lw=2, ls=ls, color=color, label=f'{label} (AUC={auc:.3f})')

ax.plot([0,1],[0,1],'k--', alpha=0.3, label='Random')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('Persistent Drift Detection (5% FDI)\nLayer 2: CUSUM vs Reconstruction-Based')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Section 11: Cross-attack and feature-specific evaluation

This section consolidates performance across detector families. It answers two principal questions:

1. Does a detector perform best on the attack mechanism it was designed to detect?
2. Which railway measurements consistently exhibit the lowest detectability?

In [ ]:
summary_rows = []
for pct in [0.02,0.05,0.10]:
    pct_label=f'{int(pct*100)}%'
    for detector in ['Isolation Forest','VAE with MAD thresholding']:
        r=all_static[pct][detector]
        summary_rows.append({'Framework':'Static','Detector':detector,
            'Attack Type':'Mixed statistical','FDI %':pct_label,
            'Precision':r['precision'],'Recall':r['recall'],'F1':r['f1'],'Accuracy':r['accuracy']})
    for attack_type in ['persistent','one_shot']:
        lr=next(iter(temporal_results[attack_type][pct].values()))
        tr=next(iter(trans_results[attack_type][pct].values()))
        for detector,r in [('LSTM-VAE with MAD thresholding',lr),
                           ('Transformer-VAE with MAD thresholding',tr)]:
            summary_rows.append({'Framework':'Temporal','Detector':detector,
                'Attack Type':attack_type.replace('_',' ').title(),'FDI %':pct_label,
                'Precision':r['precision'],'Recall':r['recall'],'F1':r['f1'],'Accuracy':r['accuracy']})
summary_df=pd.DataFrame(summary_rows)
pd.set_option('display.float_format','{:.3f}'.format)
display(summary_df)


In [ ]:
# Publication-oriented F1 comparison across attack prevalence
plot_df=summary_df.copy()
g=sns.catplot(data=plot_df,x='FDI %',y='F1',hue='Detector',col='Attack Type',
              kind='point',height=4,aspect=1.05,sharey=True)
g.set(ylim=(0,1)); g.set_axis_labels('Injected sample proportion','F1-score')
g.fig.subplots_adjust(top=0.82)
g.fig.suptitle('Detector Performance Across Attack Prevalence')
plt.show()


### Figure-selection note

The second grouped bar chart was removed because it duplicated the attack-intensity comparison above. The canonical point plot and final results table should be used in the manuscript.


In [ ]:
# McNemar test removed.
# Static and temporal models operate on different datasets (WRM_F vs RawD)
# and cannot be directly compared via McNemar's test.
# Cross-model comparison is handled via ROC-AUC in Section 11.
pass


### Removed visualization

The radar chart was removed because it duplicated metrics already reported in the canonical results table and made comparisons harder to interpret.


In [ ]:
# ── ROC — all models at 5% FDI ─────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

# Static VAE (raw reconstruction error)
vae_s_roc = vae_reconstruction_error(enc_s, dec_s, X_inj5, fw_static)
fpr, tpr, _ = roc_curve(y_true5, vae_s_roc)
ax.plot(fpr, tpr, lw=2, color='steelblue',
        label=f'Static VAE (AUC={roc_auc_score(y_true5, vae_s_roc):.3f})')

# Static IF
if_sc = -iso_s.score_samples(X_inj5)
fpr2, tpr2, _ = roc_curve(y_true5, if_sc)
ax.plot(fpr2, tpr2, lw=2, ls='--', color='gray',
        label=f'Static IF (AUC={roc_auc_score(y_true5, if_sc):.3f})')

# LSTM-VAE persistent
fpr3, tpr3, _ = roc_curve(y_p5, err_p5)
ax.plot(fpr3, tpr3, lw=2, color='crimson',
        label=f'LSTM-VAE Persistent (AUC={roc_auc_score(y_p5, err_p5):.3f})')

# LSTM-VAE one-shot
fpr4, tpr4, _ = roc_curve(y_o5, err_o5)
ax.plot(fpr4, tpr4, lw=2, ls='-.', color='darkorange',
        label=f'LSTM-VAE One-Shot (AUC={roc_auc_score(y_o5, err_o5):.3f})')

# Transformer-VAE persistent
fpr5, tpr5, _ = roc_curve(y_tp5, err_tp5)
ax.plot(fpr5, tpr5, lw=2, ls=':', color='darkgreen',
        label=f'Transformer-VAE Persistent (AUC={roc_auc_score(y_tp5, err_tp5):.3f})')

# Transformer-VAE one-shot
fpr6, tpr6, _ = roc_curve(y_to5, err_to5)
ax.plot(fpr6, tpr6, lw=2, ls=(0,(3,1,1,1)), color='purple',
        label=f'Transformer-VAE One-Shot (AUC={roc_auc_score(y_to5, err_to5):.3f})')

ax.plot([0,1],[0,1],'k--', alpha=0.4)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — 5% FDI Intensity')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# Bootstrap 95% confidence intervals for principal 5% experiments
from sklearn.utils import resample

def bootstrap_auc_f1(y_true,scores,threshold,n_boot=2000,seed=SEED):
    rng=np.random.RandomState(seed); aucs=[]; f1s=[]
    for _ in range(n_boot):
        sample_idx=resample(np.arange(len(y_true)),random_state=rng)
        yt=np.asarray(y_true)[sample_idx]; sc=np.asarray(scores)[sample_idx]
        if len(np.unique(yt))<2: continue
        aucs.append(roc_auc_score(yt,sc))
        f1s.append(f1_score(yt,(sc>threshold).astype(int),zero_division=0))
    return {'AUC':np.mean(aucs),'AUC_CI_Lower':np.percentile(aucs,2.5),
            'AUC_CI_Upper':np.percentile(aucs,97.5),'F1':np.mean(f1s),
            'F1_CI_Lower':np.percentile(f1s,2.5),'F1_CI_Upper':np.percentile(f1s,97.5)}

_train_static_err=vae_reconstruction_error(enc_s,dec_s,X_static,fw_static)
_train_static_score=mad_score_from_train(_train_static_err,_train_static_err)
_test_static_err=vae_reconstruction_error(enc_s,dec_s,X_inj5,fw_static)
_test_static_score=mad_score_from_train(_train_static_err,_test_static_err)
_train_lstm_err=temporal_reconstruction_error(lstm_enc,lstm_dec,X_train_t)
_train_lstm_score=mad_score_from_train(_train_lstm_err,_train_lstm_err)
_train_transformer_err=temporal_reconstruction_error(trans_enc,trans_dec,X_train_t)
_train_transformer_score=mad_score_from_train(_train_transformer_err,_train_transformer_err)

ci_specs=[
('VAE with MAD thresholding — Mixed statistical 5%',y_true5,_test_static_score,np.percentile(_train_static_score,95)),
('LSTM-VAE with MAD thresholding — Persistent 5%',y_p5,mad_score_from_train(_train_lstm_err,err_p5),np.percentile(_train_lstm_score,95)),
('Transformer-VAE with MAD thresholding — Persistent 5%',y_tp5,mad_score_from_train(_train_transformer_err,err_tp5),np.percentile(_train_transformer_score,95)),
('LSTM-VAE with MAD thresholding — One-shot 5%',y_o5,mad_score_from_train(_train_lstm_err,err_o5),np.percentile(_train_lstm_score,95)),
('Transformer-VAE with MAD thresholding — One-shot 5%',y_to5,mad_score_from_train(_train_transformer_err,err_to5),np.percentile(_train_transformer_score,95))]
ci_rows=[]
for label,y,sc,thr in ci_specs:
    row={'Experiment':label}; row.update(bootstrap_auc_f1(y,sc,thr)); ci_rows.append(row)
confidence_intervals=pd.DataFrame(ci_rows)
display(confidence_intervals.round(3))


In [ ]:
# Canonical final evaluation table
canonical_rows=[]
def record_result(family,variant,detector,y,score,threshold,higher_is_anomaly=True):
    y=np.asarray(y); score=np.asarray(score)
    if higher_is_anomaly:
        pred=(score>threshold).astype(int); auc_score=score
    else:
        pred=(score<threshold).astype(int); auc_score=-score
    canonical_rows.append({'Attack Family':family,'Attack Variant':variant,'Detector':detector,
        'Threshold':float(threshold),'Precision':precision_score(y,pred,zero_division=0),
        'Recall':recall_score(y,pred,zero_division=0),'F1':f1_score(y,pred,zero_division=0),
        'Accuracy':np.mean(y==pred),'ROC_AUC':roc_auc_score(y,auc_score),
        'Attacked Samples':int(y.sum()),'Total Samples':int(len(y))})

_train_if=-iso_s.score_samples(X_static); _test_if=-iso_s.score_samples(X_inj5)
record_result('Statistical','Mixed statistical 5%','Isolation Forest',y_true5,_test_if,np.percentile(_train_if,95))
record_result('Statistical','Mixed statistical 5%','VAE with MAD thresholding',y_true5,_test_static_score,np.percentile(_train_static_score,95))
record_result('Temporal','Persistent 5%','LSTM-VAE with MAD thresholding',y_p5,mad_score_from_train(_train_lstm_err,err_p5),np.percentile(_train_lstm_score,95))
record_result('Temporal','One-shot 5%','LSTM-VAE with MAD thresholding',y_o5,mad_score_from_train(_train_lstm_err,err_o5),np.percentile(_train_lstm_score,95))
record_result('Temporal','Persistent 5%','Transformer-VAE with MAD thresholding',y_tp5,mad_score_from_train(_train_transformer_err,err_tp5),np.percentile(_train_transformer_score,95))
record_result('Temporal','One-shot 5%','Transformer-VAE with MAD thresholding',y_to5,mad_score_from_train(_train_transformer_err,err_to5),np.percentile(_train_transformer_score,95))
record_result('Temporal','Persistent 5%','CUSUM baseline',y_p5,test_cu_p5,np.percentile(train_cu,95))
record_result('Replay','Exact replay 5%','Stagnation detector',y_r5,test_stag,np.percentile(train_stag,5),False)

master_results=pd.DataFrame(canonical_rows).sort_values(['Attack Family','Attack Variant','F1'],ascending=[True,True,False]).reset_index(drop=True)
display(master_results.round(3))
attack_family_table=master_results.pivot_table(index='Detector',columns='Attack Family',values='F1',aggfunc='mean')
display(attack_family_table.round(3))
plt.figure(figsize=(9,4.8))
sns.heatmap(attack_family_table,annot=True,fmt='.3f',vmin=0,vmax=1,cmap='YlGnBu')
plt.title('Attack-Family-Specific Detector Performance (F1)')
plt.xlabel('Attack family'); plt.ylabel('Detector'); plt.tight_layout(); plt.show()


## Section 12: Export models and analysis results

The export stage saves model artefacts, scalers and reproducible CSV tables. Operational cybersecurity implications should be discussed from these measured results; no arbitrary exposure score is created.

In [ ]:
import os, pickle
SAVE_DIR=BASE_PATH+'FDI_Attack_Aware_Results_Cleaned/'
os.makedirs(SAVE_DIR,exist_ok=True)
vae_s.save(SAVE_DIR+'static_vae.keras')
if 'lstm_vae' in globals(): lstm_vae.save(SAVE_DIR+'lstm_vae.keras')
if 'trans_vae' in globals(): trans_vae.save(SAVE_DIR+'transformer_vae.keras')
with open(SAVE_DIR+'scaler_static.pkl','wb') as f: pickle.dump(scaler_static,f)
if 'scaler_t' in globals():
    with open(SAVE_DIR+'scaler_temporal.pkl','wb') as f: pickle.dump(scaler_t,f)
data_audit.to_csv(SAVE_DIR+'data_audit.csv',index=False)
summary_df.to_csv(SAVE_DIR+'attack_intensity_results.csv',index=False)
master_results.to_csv(SAVE_DIR+'canonical_master_results.csv',index=False)
attack_family_table.to_csv(SAVE_DIR+'attack_family_specific_f1.csv')
confidence_intervals.to_csv(SAVE_DIR+'bootstrap_confidence_intervals.csv',index=False)
print(f'Canonical outputs saved to: {SAVE_DIR}')


## Reporting guide

Use the following outputs in the manuscript:

1. data-quality audit and leakage-safe preprocessing;
2. clean-data reconstruction behaviour and low-dimensional visualizations;
3. statistical detection performance across 2%, 5%, and 10% injection;
4. temporal persistent and one-shot attack performance;
5. exact and perturbed replay results;
6. feature-specific susceptibility heatmaps;
7. bootstrap confidence intervals; and
8. the canonical attack-family-specific detector comparison.

### Interpretation boundaries

- Do not present MAD as a separate model.
- Do not describe CUSUM as a proposed detector.
- Do not call the final heatmap a complete cross-attack generalization study.
- Do not claim adversarial robustness beyond the explicitly tested replay perturbations.
- Do not create arbitrary high/medium/low cyber-exposure categories.
